# Phase 3 Experiments - Phase E: Sensitivity Analysis

This notebook tests the **best 2 pruning methods** from Phase C+D at different sparsity levels (30% and 70% actual overall sparsity).

**Purpose**: Show accuracy-efficiency trade-off curve (Pareto frontier) for publication.

**DEPENDENCY**: Phase C+D must complete first to identify the best 2 pruning methods!

**Runs**: 4 total
- E1: Best Method #1 at 30% actual sparsity (target=0.39)
- E2: Best Method #1 at 70% actual sparsity (target=0.91)
- E3: Best Method #2 at 30% actual sparsity (target=0.39)
- E4: Best Method #2 at 70% actual sparsity (target=0.91)

**NOTE**: Update `BEST_METHOD_1` and `BEST_METHOD_2` variables below after Phase C+D results!

**SPARSITY NOTE**: Target sparsity is higher than actual because pruning only affects Linear layers (~60% of model). To achieve X% actual overall sparsity, we set target to approximately X/0.77.</cell id="cell-0">
<parameter name="cell_type">markdown

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q iterative-stratification scikit-learn pandas numpy tqdm

In [ ]:
# Setup logging
import sys
from datetime import datetime

class Logger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger("/kaggle/working/experiment_log_sensitivity.txt")
print(f"Experiment started at: {datetime.now()}")
print("Phase E: Sensitivity Analysis (30% and 70% sparsity)")

In [ ]:
# Clone repository
!git clone https://github.com/SaifSiddique009/kd_pruning_quantization_framework_for_nlp.git
%cd kd_pruning_quantization_framework_for_nlp
!git checkout phase3-comprehensive-experiments
!git log -1 --oneline

---
## Configuration: UPDATE AFTER PHASE C+D RESULTS
---

In [ ]:
# ============================================================================
# UPDATE THESE VALUES AFTER PHASE C+D COMPLETES!
# ============================================================================

# Best pruning methods from Phase C+D (options: magnitude, wanda, gradual, structured)
# Choose the 2 methods with highest F1 retention after pruning
BEST_METHOD_1 = "magnitude"  # UPDATE: Replace with best method from Phase C+D
BEST_METHOD_2 = "wanda"      # UPDATE: Replace with second best method from Phase C+D

# Best KD model to use for sensitivity analysis
# Choose the KD model that performed best with pruning
BEST_KD_MODEL = "Saif-Siddique/bangla-cyberbully-kd1-xlmroberta-to-sahajbert"
BEST_KD_EVAL_FOLD = 3  # Use the eval_fold for this KD model's teacher

# Sparsity levels to test
# NOTE: Target sparsity is set higher to achieve desired ACTUAL overall sparsity
# because pruning only affects Linear layers (~60% of model weights)
# Formula: target ≈ actual / 0.77 (where 0.77 accounts for Linear layer proportion)
SPARSITY_LOW = 0.39   # Target 39% → ~30% actual overall sparsity
SPARSITY_HIGH = 0.91  # Target 91% → ~70% actual overall sparsity

# Dataset path
DATASET_PATH = "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv"
AUTHOR = "Saif-Siddique"

print(f"Configuration:")
print(f"  Best Method #1: {BEST_METHOD_1}")
print(f"  Best Method #2: {BEST_METHOD_2}")
print(f"  Best KD Model: {BEST_KD_MODEL}")
print(f"  Eval Fold: {BEST_KD_EVAL_FOLD}")
print(f"  Sparsity Targets: {SPARSITY_LOW*100:.0f}% (→~30% actual), {SPARSITY_HIGH*100:.0f}% (→~70% actual)")

---
## E1: Best Method #1 at 30% Sparsity
---

In [ ]:
import subprocess

print(f"\n{'='*60}")
print(f"E1: {BEST_METHOD_1} at {SPARSITY_LOW*100:.0f}% sparsity")
print(f"{'='*60}\n")

cmd = f"""python main.py \
    --dataset_path "{DATASET_PATH}" \
    --author_name "{AUTHOR}" \
    --pipeline prune_only \
    --teacher_checkpoint "{BEST_KD_MODEL}" \
    --prune_method {BEST_METHOD_1} \
    --prune_sparsity {SPARSITY_LOW} \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold {BEST_KD_EVAL_FOLD} \
    --output_dir ./results/sensitivity/{BEST_METHOD_1}_30pct"""

print(f"Running: {cmd}\n")
!{cmd}

---
## E2: Best Method #1 at 70% Sparsity
---

In [ ]:
print(f"\n{'='*60}")
print(f"E2: {BEST_METHOD_1} at {SPARSITY_HIGH*100:.0f}% sparsity")
print(f"{'='*60}\n")

cmd = f"""python main.py \
    --dataset_path "{DATASET_PATH}" \
    --author_name "{AUTHOR}" \
    --pipeline prune_only \
    --teacher_checkpoint "{BEST_KD_MODEL}" \
    --prune_method {BEST_METHOD_1} \
    --prune_sparsity {SPARSITY_HIGH} \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold {BEST_KD_EVAL_FOLD} \
    --output_dir ./results/sensitivity/{BEST_METHOD_1}_70pct"""

print(f"Running: {cmd}\n")
!{cmd}

---
## E3: Best Method #2 at 30% Sparsity
---

In [ ]:
print(f"\n{'='*60}")
print(f"E3: {BEST_METHOD_2} at {SPARSITY_LOW*100:.0f}% sparsity")
print(f"{'='*60}\n")

cmd = f"""python main.py \
    --dataset_path "{DATASET_PATH}" \
    --author_name "{AUTHOR}" \
    --pipeline prune_only \
    --teacher_checkpoint "{BEST_KD_MODEL}" \
    --prune_method {BEST_METHOD_2} \
    --prune_sparsity {SPARSITY_LOW} \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold {BEST_KD_EVAL_FOLD} \
    --output_dir ./results/sensitivity/{BEST_METHOD_2}_30pct"""

print(f"Running: {cmd}\n")
!{cmd}

---
## E4: Best Method #2 at 70% Sparsity
---

In [ ]:
print(f"\n{'='*60}")
print(f"E4: {BEST_METHOD_2} at {SPARSITY_HIGH*100:.0f}% sparsity")
print(f"{'='*60}\n")

cmd = f"""python main.py \
    --dataset_path "{DATASET_PATH}" \
    --author_name "{AUTHOR}" \
    --pipeline prune_only \
    --teacher_checkpoint "{BEST_KD_MODEL}" \
    --prune_method {BEST_METHOD_2} \
    --prune_sparsity {SPARSITY_HIGH} \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold {BEST_KD_EVAL_FOLD} \
    --output_dir ./results/sensitivity/{BEST_METHOD_2}_70pct"""

print(f"Running: {cmd}\n")
!{cmd}

---
## Final Status Check
---

In [ ]:
import os
import json
from datetime import datetime

print(f"\n{'='*70}")
print(f"SENSITIVITY ANALYSIS COMPLETION STATUS - {datetime.now()}")
print(f"{'='*70}\n")

experiments = [
    (f"E1 {BEST_METHOD_1} @ 30%", f"./results/sensitivity/{BEST_METHOD_1}_30pct", 0.3),
    (f"E2 {BEST_METHOD_1} @ 70%", f"./results/sensitivity/{BEST_METHOD_1}_70pct", 0.7),
    (f"E3 {BEST_METHOD_2} @ 30%", f"./results/sensitivity/{BEST_METHOD_2}_30pct", 0.3),
    (f"E4 {BEST_METHOD_2} @ 70%", f"./results/sensitivity/{BEST_METHOD_2}_70pct", 0.7),
]

success_count = 0
results_data = []

print(f"{'Experiment':<25} {'Sparsity':<10} {'F1 Weighted':<12} {'F1 Macro':<12} {'Status'}")
print("-" * 75)

for name, output_dir, sparsity in experiments:
    json_path = os.path.join(output_dir, "results_final.json")
    
    if os.path.exists(json_path):
        with open(json_path) as f:
            data = json.load(f)
        
        # Get final metrics
        if isinstance(data, list):
            final_metrics = data[-1]
        elif 'metrics' in data:
            final_metrics = data['metrics'][-1]
        else:
            final_metrics = data
        
        f1_weighted = final_metrics.get('f1_weighted', 'N/A')
        f1_macro = final_metrics.get('f1_macro', 'N/A')
        actual_sparsity = final_metrics.get('sparsity_percent', 'N/A')
        
        if isinstance(f1_weighted, (int, float)) and isinstance(f1_macro, (int, float)):
            print(f"{name:<25} {sparsity*100:<10.0f}% {f1_weighted:<12.4f} {f1_macro:<12.4f} SUCCESS")
            results_data.append({
                'experiment': name,
                'target_sparsity': sparsity,
                'actual_sparsity': actual_sparsity,
                'f1_weighted': f1_weighted,
                'f1_macro': f1_macro
            })
        else:
            print(f"{name:<25} {sparsity*100:<10.0f}% {str(f1_weighted):<12} {str(f1_macro):<12} SUCCESS")
        success_count += 1
    else:
        print(f"{name:<25} {sparsity*100:<10.0f}% {'N/A':<12} {'N/A':<12} FAILED")

print("-" * 75)
print(f"\nCompleted: {success_count}/{len(experiments)} experiments")
print(f"{'='*70}")

---
## Generate Sensitivity Comparison Table
---

In [ ]:
import pandas as pd

if results_data:
    df = pd.DataFrame(results_data)
    
    print("\nSensitivity Analysis Results:")
    print("="*70)
    print(df.to_markdown(index=False))
    
    # Save to CSV
    df.to_csv('./results/sensitivity/sensitivity_comparison.csv', index=False)
    print("\nSaved to: ./results/sensitivity/sensitivity_comparison.csv")
    
    # Create pivot table for visualization
    print("\n\nF1 Weighted by Method and Sparsity:")
    print("="*50)
    
    # Simple comparison
    for method in [BEST_METHOD_1, BEST_METHOD_2]:
        method_data = [d for d in results_data if method in d['experiment']]
        if method_data:
            print(f"\n{method.upper()}:")
            for d in method_data:
                print(f"  {d['target_sparsity']*100:.0f}% sparsity: F1={d['f1_weighted']:.4f}")
else:
    print("No results to display. Check if experiments completed successfully.")

In [ ]:
# Copy results to output
!cp -r ./results /kaggle/working/
!ls -la /kaggle/working/results/sensitivity/

In [ ]:
print(f"\nSensitivity Analysis notebook completed at: {datetime.now()}")
print("\nDownload results from /kaggle/working/results/sensitivity/")
print("\nKey files:")
print("  - sensitivity_comparison.csv: Summary table")
print(f"  - {BEST_METHOD_1}_30pct/results_final.json")
print(f"  - {BEST_METHOD_1}_70pct/results_final.json")
print(f"  - {BEST_METHOD_2}_30pct/results_final.json")
print(f"  - {BEST_METHOD_2}_70pct/results_final.json")